# 03 — Prepare Recognition Crops

**Purpose:** Extract cropped region images for the text recognition model.

The recognizer learns from `(image_crop, text_label)` pairs — not full pages.
This notebook:
1. Runs the crop preparation pipeline.
2. Validates the output crops.
3. Displays a contact sheet of random crops for visual inspection.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ── Check that the train/val splits exist ─────────────────────────
from src.utils.paths import get_path

train_split = get_path('train_split')
val_split   = get_path('val_split')

if not train_split.exists():
    print('ERROR: train_split.jsonl not found.')
    print('Run: python -m src.data.make_splits first.')
else:
    print(f'✅ train_split.jsonl  →  {train_split}')
    
if not val_split.exists():
    print('ERROR: val_split.jsonl not found.')
else:
    print(f'✅ val_split.jsonl    →  {val_split}')

In [ ]:
# ── Prepare TRAIN crops ───────────────────────────────────────────
# This reads train_split.jsonl, opens each image, and saves crops.
# Skips non-Ukrainian, illegible, image, graph, and empty-text regions.

from src.utils.jsonl import read_jsonl
from src.recognition.prepare_crops import prepare_crops_from_records

records     = read_jsonl(train_split)
images_root = get_path('raw_train_images')
out_img_dir = get_path('crops_train_img')
out_csv     = get_path('crops_train_csv')

print(f'Processing {len(records):,} train records...')
n_train = prepare_crops_from_records(records, images_root, out_img_dir, out_csv)
print(f'\n✅ Train crops ready: {n_train:,}')

In [ ]:
# ── Prepare VAL crops ─────────────────────────────────────────────

records_val = read_jsonl(val_split)
out_img_val = get_path('crops_val_img')
out_csv_val = get_path('crops_val_csv')

print(f'Processing {len(records_val):,} val records...')
n_val = prepare_crops_from_records(records_val, images_root, out_img_val, out_csv_val)
print(f'\n✅ Val crops ready: {n_val:,}')

In [ ]:
# ── Validate crops ────────────────────────────────────────────────

from src.recognition.validate_crops import validate_crops, print_summary, save_report

summary_train = validate_crops(out_csv, n_contact=20)
print_summary(summary_train, 'train')
save_report(summary_train, 'train')

summary_val = validate_crops(out_csv_val, n_contact=20)
print_summary(summary_val, 'val')
save_report(summary_val, 'val')

In [ ]:
# ── Display the contact sheet ─────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

contact_path = summary_train.get('contact_sheet')
if contact_path and Path(contact_path).exists():
    img = mpimg.imread(contact_path)
    plt.figure(figsize=(16, 10))
    plt.imshow(img)
    plt.title('Recognition Crop Contact Sheet — verify crops look readable', fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Contact sheet not generated (may need opencv).')

In [ ]:
# ── Inspect a few crops individually ─────────────────────────────
import pandas as pd
from PIL import Image

df = pd.read_csv(out_csv)
print(f'Train labels.csv: {len(df):,} rows')
print(f'Columns: {df.columns.tolist()}')
print(f'\nType distribution:')
print(df['type'].value_counts())
print(f'\nSource distribution:')
print(df['source'].value_counts())

In [ ]:
# ── Display 6 random crops with labels ───────────────────────────

sample_df = df.sample(min(6, len(df)), random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(16, 6))
axes = axes.flatten()

for ax, (_, row) in zip(axes, sample_df.iterrows()):
    crop_path = Path(row['crop_path'])
    if crop_path.exists():
        img = Image.open(crop_path)
        ax.imshow(img, cmap='gray' if img.mode == 'L' else None)
    else:
        ax.text(0.5, 0.5, 'File not found', ha='center', va='center')
    text = row['text']
    label = (text[:35] + '…') if len(text) > 35 else text
    ax.set_title(f'[{row["type"]}] {label}', fontsize=8, wrap=True)
    ax.axis('off')

for ax in axes[len(sample_df):]:
    ax.axis('off')

plt.suptitle('Random Recognition Crops — check legibility and label accuracy', fontsize=11)
plt.tight_layout()
plt.show()

## ✅ Next Step

```bash
# Prepare YOLO detection dataset:
python -m src.detection.convert_to_yolo
python -m src.detection.validate_yolo

# Then open:
# notebooks/04_prepare_yolo.ipynb
```